Story Building: Why are we doing this lab?

Imagine you are working as a junior data engineer in an e-commerce company called Maven Fuzzy Factory.
This company sells products online. Every day, many business systems generate data:
    the orders system records customer purchases
    the order items system records which products are inside each order
    the refund system records returned or refunded items
    the product master stores product details
    the website session tracking system records user visits
    the pageview tracking system records which pages users visited

Now, all this data is periodically delivered to the data engineering team as a ZIP package.
That ZIP file is placed in a folder called the landing zone.

At this stage, the files are only received, not yet trusted, not yet transformed, and not yet integrated.

Your job in this lab is not to analyze the business.
Your first responsibility is much more basic and much more important:

Receive the source files safely, unpack them, register what arrived, attach ingestion metadata, and move them into the raw layer.

This is the first step of a real data pipeline.

What exactly is the problem statement?

Problem Statement: Maven Fuzzy Factory provides a compressed dataset package containing multiple structured CSV files related to orders, products, 
refunds, sessions, and pageviews. The data engineering team must design a batch ingestion workflow that reads the dataset package from a landing zone, 
extracts all structured files, ingests them into the pipeline, adds audit metadata, and stores them in a raw data zone for further downstream validation and processing.

The workflow must:
    read the ZIP file from the landing zone
    extract all CSV files
    identify each table automatically
    ingest each file into Python
    attach ingestion audit details
    write each ingested table into a raw zone
    generate an ingestion summary report

Why is this important in real data engineering?
Think data engineering starts from analysis, dashboards, or machine learning.
Actually, before any of that, there is a very important responsibility:

Can you receive source data in a controlled and traceable way?
    If the answer is no, then everything later becomes unreliable.

This lab teaches the foundation of pipeline discipline:
    where data arrives
    how raw source files are handled
    how data lineage begins
    how audit tracking starts
    how a pipeline becomes reproducible

What is the meaning of Landing Zone and Raw Zone?
Landing Zone
The landing zone is the place where source data first arrives.
Think of it like a courier receiving area in a company.
In this lab:
landing/ contains the ZIP file
the ZIP file is the exact package received from the source

At this stage:
no cleaning is done
no schema check is done
no joins are done
no business rules are applied

It is simply the arrival point.
    
Raw Zone
The raw zone is the first controlled storage layer after ingestion.
Think of it like this:
landing zone = package received
raw zone = package opened, contents registered, basic metadata attached, safely stored
In the raw zone:
data is still mostly original
we do not yet transform business meaning
we preserve source truth as much as possible
we add pipeline metadata like:
which file it came from when it was ingested

This helps later in:
    traceability
    auditing
    debugging
    reprocessing

In [ ]:
What data is present in this dataset?

The ZIP contains multiple business tables. Each one represents a part of the e-commerce system.
orders.csv
Contains order-level information.

Typical meaning:
one row = one customer order

Possible columns:
    order_id
    created_at
    website_session_id
    user_id
    primary_product_id
    items_purchased
    price_usd
    cogs_usd
Business meaning:
This table tells us when an order happened, who placed it, what primary product is linked, how many items were purchased, and revenue/cost information.

order_items.csv
Contains item-level detail for each order.

Typical meaning:
one order can have multiple order items
Business meaning:
This table is more detailed than orders.csv. It helps answer which exact products were bought inside each order.

order_item_refunds.csv
Contains refund details.
Business meaning:
Some items were refunded after purchase. This table records refunded items and refund amounts.

products.csv
Contains product master information.
Business meaning:
This table helps us interpret product IDs into business-readable product details.

website_sessions.csv
Contains user visit sessions.
Business meaning:
Each row is a website session. It helps track marketing source, device type, and user session behavior.

website_pageviews.csv
Contains pageview events.
Business meaning:
This is a more granular behavior table. It tells which pages were visited in a session.

maven_fuzzy_factory_data_dictionary.csv
Contains metadata.
Business meaning:
This file explains the table names and field names. It is useful later for schema validation and understanding the dataset.

In [ ]:
What exactly are you are expected to do?

You must create an ingestion workflow that:
    takes the ZIP package from landing zone
    unpacks it
    scans all extracted CSV files
    reads each file into a DataFrame
    adds ingestion metadata
    writes each table into raw zone
    prepares a summary report of what was ingested

In [ ]:
Folder structure and why it exists
project/
│
├── landing/
│   └── Maven+Fuzzy+Factory.zip
├── extracted/
├── raw/
└── logs/

Meaning of each folder
landing/
Where the source ZIP arrives.

extracted/
Temporary zone where the ZIP contents are unpacked.

raw/
Where the ingested raw versions of the CSV files are stored.

logs/
Where execution logs are stored so that we know what happened during the run.

In [ ]:
#Step 1: Import required modules
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime
import pandas as pd
import logging

In [ ]:
# Step 2: Define folder paths
# ---------- folder setup ----------
landing_dir = Path("D:\\DE-Lab\\project\\landing")
extracted_dir = Path("D:\\DE-Lab\\project\\extracted")
raw_dir = Path("D:\\DE-Lab\\project\\raw")
logs_dir = Path("D:\\DE-Lab\\project\\logs")

In [ ]:
# Step 3: Create required folders if they do not exist
for d in [extracted_dir, raw_dir, logs_dir]:
    d.mkdir(exist_ok=True)

In [ ]:
# Step 4: Configure logging
logging.basicConfig(
    filename=logs_dir / "lab1_ingestion.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [ ]:
# Point to the ZIP file in the landing zone and extract zip and write into logging
zip_path = landing_dir / "Maven+Fuzzy+Factory.zip"
with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extracted_dir)
logging.info("ZIP extracted successfully")

In [ ]:
# Identify all CSV files in the extracted folder and raise INFO if no csv is present
csv_files = list(extracted_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV files found after extraction.")

In [ ]:
# Capture ingestion timestamp
ingestion_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# Create a summary container
summary = []

In [ ]:
# Core ingestion loop
for file in csv_files:
    df = pd.read_csv(file)

    df["source_file"] = file.name
    df["ingestion_time"] = ingestion_time

    output_file = raw_dir / f"raw_{file.name}"
    df.to_csv(output_file, index=False)

    summary.append({
        "table_name": file.stem,
        "rows_ingested": len(df),
        "columns_ingested": len(df.columns),
        "raw_output_file": output_file.name
    })

    logging.info(f"Ingested {file.name} with {len(df)} rows")

In [ ]:
# Loop through each CSV file
for file in csv_files:
    df = pd.read_csv(file)

    # Add audit columns
    df["source_file"] = file.name
    df["ingestion_time"] = ingestion_time

    output_file = raw_dir / f"raw_{file.name}"
    df.to_csv(output_file, index=False)

    summary.append({
        "table_name": file.stem,
        "rows_ingested": len(df),
        "columns_ingested": len(df.columns),
        "raw_output_file": output_file.name
    })

    logging.info(f"Ingested {file.name} with {len(df)} rows")

In [ ]:
summary_df = pd.DataFrame(summary)
summary_df.to_csv(raw_dir / "raw_ingestion_summary.csv", index=False)

print(summary_df)

In [3]:
# from pathlib import Path
# from zipfile import ZipFile
# from datetime import datetime
# import pandas as pd
# import logging

# # ---------- folder setup ----------
# landing_dir = Path("D:\\DE-Lab\\project\\landing")
# extracted_dir = Path("D:\\DE-Lab\\project\\extracted")
# raw_dir = Path("D:\\DE-Lab\\project\\raw")
# logs_dir = Path("D:\\DE-Lab\\project\\logs")



# # ---------- logging ----------
# logging.basicConfig(
#     filename=logs_dir / "lab1_ingestion.log",
#     level=logging.INFO,
#     format="%(asctime)s - %(levelname)s - %(message)s"
# )

# zip_path = landing_dir / "Maven+Fuzzy+Factory.zip"
# print("zip_path: ",zip_path)

# # ---------- extract ----------
# with ZipFile(zip_path, "r") as zip_ref:
#     zip_ref.extractall(extracted_dir)

# logging.info("ZIP extracted successfully")

# # ---------- ingest all csv files ----------
# csv_files = list(extracted_dir.glob("*.csv"))

# if not csv_files:
#     raise FileNotFoundError("No CSV files found after extraction.")

# ingestion_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
# summary = []

# for file in csv_files:
#     df = pd.read_csv(file)

#     # Add audit columns
#     df["source_file"] = file.name
#     df["ingestion_time"] = ingestion_time

#     output_file = raw_dir / f"raw_{file.name}"
#     df.to_csv(output_file, index=False)

#     summary.append({
#         "table_name": file.stem,
#         "rows_ingested": len(df),
#         "columns_ingested": len(df.columns),
#         "raw_output_file": output_file.name
#     })

#     logging.info(f"Ingested {file.name} with {len(df)} rows")

# summary_df = pd.DataFrame(summary)
# summary_df.to_csv(raw_dir / "raw_ingestion_summary.csv", index=False)

# print(summary_df)

zip_path:  D:\DE-Lab\project\landing\Maven+Fuzzy+Factory.zip
                            table_name  rows_ingested  columns_ingested  \
0  maven_fuzzy_factory_data_dictionary             36                 5   
1                               orders          32313                10   
2                          order_items          40025                 9   
3                   order_item_refunds           1731                 7   
4                             products              4                 5   
5                    website_pageviews        1188124                 6   
6                     website_sessions         472871                11   

                               raw_output_file  
0  raw_maven_fuzzy_factory_data_dictionary.csv  
1                               raw_orders.csv  
2                          raw_order_items.csv  
3                   raw_order_item_refunds.csv  
4                             raw_products.csv  
5                    raw_website_pageviews.c